[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 困难: INT8 量化线性层

实现一个**训练后量化（PTQ - Post-Training Quantization）的线性层**，使用 INT8 权重。

### 函数签名
```python
class Int8Linear(nn.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### 量化（按通道）
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. 存储为 `register_buffer`（不可训练）
4. 前向：反量化（`int8.float() * scale`）然后矩阵乘法

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ 在此实现你的代码

class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        pass  # 量化权重, 注册 buffers

    def forward(self, x):
        pass  # 反量化和矩阵乘法

### AWQ vs PTQ vs GPTQ 详细对比

这三种都是**量化方法**，但复杂度和优化策略完全不同。让我详细解释：

---

### 1. PTQ (Post-Training Quantization) - 训练后量化

#### 定义
训练完成后，直接对模型权重进行量化，**不需要额外训练或微调**。

#### 方法分类

**最简单的PTQ：**
```python
# Min-Max量化
scale = (weight.max() - weight.min()) / 255
# 或
scale = weight.abs().max() / 127  # 对称量化
```

**稍微复杂点的PTQ：**
```python
# 使用KL散度寻找最优阈值（如TensorRT的做法）
# 在验证集上运行，统计激活值分布，找到最佳截断点
best_threshold = find_kl_divergence_threshold(activation_distribution)
```

#### 特点
- ✅ **不需要数据**（或仅需少量校准数据）
- ✅ **计算快**，几秒钟完成
- ✅ **实现简单**
- ❌ **精度损失较大**（尤其是大模型）
- ❌ **没有考虑激活值分布**

---

### 2. AWQ (Activation-aware Weight Quantization) - 2023

#### 核心思想
**"不是所有权重都同等重要，应该保护对激活值敏感的通道"**

#### 关键洞察
通过分析**激活值的分布**，找出哪些权重通道对输出影响最大，为这些通道保留更高精度。

#### 工作原理
```python
# 伪代码
def awq_quantize(weight, calibration_data):
    # 1. 用校准数据前向传播，收集激活值
    activations = []
    for batch in calibration_data:
        act = model(batch)  # 记录中间层激活值
        activations.append(act)
    
    # 2. 计算每个通道的重要性 = 激活值的平均绝对值
    importance = torch.stack(activations).abs().mean(dim=(0, 2, 3))
    
    # 3. 对重要通道，搜索最优缩放因子
    # 目的：最小化量化误差
    scales = []
    for channel_idx in range(num_channels):
        if importance[channel_idx] > threshold:
            # 重要通道：尝试多个缩放因子，选最优
            best_scale = search_scale(weight[:, channel_idx])
        else:
            best_scale = 1.0  # 不重要通道直接量化
        scales.append(best_scale)
    
    # 4. 应用缩放后量化
    weight_scaled = weight * scales
    scale = weight_scaled.abs().max(dim=1) / 127
    weight_int8 = round(weight_scaled / scale)
```

#### 特点
- ✅ **精度高**（比普通PTQ好很多）
- ✅ **不需要训练**
- ✅ **速度快**（只需少量校准数据）
- ✅ **适用于4-bit、3-bit等低位宽**
- ❌ **需要校准数据**（~128-512个样本）
- ❌ **实现比PTQ复杂**

---

### 3. GPTQ (Generative Pre-trained Transformer Quantization) - 2023

#### 核心思想
**"逐层量化，并利用Hessian矩阵补偿量化误差"**

#### 关键洞察
量化某层的权重时，可以调整该层**未量化的权重**来补偿量化误差，使整体输出误差最小化。

#### 工作原理
```python
# 伪代码
def gptq_quantize(layer_weight, calibration_data, bits=4):
    # 1. 收集激活值，计算Hessian矩阵（二阶梯度信息）
    H = compute_hessian(calibration_data, layer_weight)  # 重要！
    
    # 2. 逐行量化（或逐块量化）
    for row in range(layer_weight.shape[0]):
        # 量化当前行的权重
        weight_row = layer_weight[row]
        quantized_row = round(weight_row / scale) * scale
        
        # 3. 误差补偿：调整未量化的权重来补偿
        error = weight_row - quantized_row
        # 使用Hessian矩阵将误差分配到未量化的权重上
        compensation = solve_least_squares(H, error)
        # 更新未量化的权重
        layer_weight[row, :] += compensation
    
    return quantized_weight
```

### 核心优势
- **误差补偿机制**：量化一部分权重时，调整其他权重来"弥补"
- **Hessian矩阵**：知道哪些权重调整后影响最小

### 特点
- ✅ **精度非常高**（接近FP16）
- ✅ **支持极低位宽**（2-bit, 3-bit）
- ✅ **适用于大模型**（LLaMA, GPT等）
- ❌ **计算量大**（需要Hessian矩阵）
- ❌ **实现复杂**
- ❌ **需要校准数据**（和AWQ类似）

---

## 对比表格

| 特性 | PTQ | AWQ | GPTQ |
|------|-----|-----|------|
| **核心策略** | 直接量化 | 保护重要通道 | 误差补偿 |
| **校准数据** | 可选（少） | 需要（~128样本） | 需要（~128样本） |
| **复杂度** | ⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **速度** | 极快 | 快 | 慢（需Hessian计算） |
| **4-bit精度** | 一般 | 好 | 优秀 |
| **3-bit精度** | 差 | 可用 | 优秀 |
| **适用场景** | 快速部署 | 平衡精度/速度 | 追求极致精度 |
| **代表模型** | TensorRT | AWQ-LLM | GPTQ-for-LLaMA |

---

## 实际应用示例

```python
# 1. PTQ - 最简单
model = torch.load('model.pth')
quantized = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

# 2. AWQ - 精度和速度的平衡
from awq import AutoAWQForCausalLM
model = AutoAWQForCausalLM.from_pretrained("llama-7b")
model.quantize(calibration_data, bits=4)  # 需要校准数据

# 3. GPTQ - 最高精度
from transformers import AutoModelForCausalLM, GPTQConfig
quantization_config = GPTQConfig(bits=4, dataset="c4")
model = AutoModelForCausalLM.from_pretrained("llama-7b", quantization_config=quantization_config)
```

---

## 选择建议

| 需求 | 推荐方法 |
|------|---------|
| 快速验证/原型 | PTQ |
| 生产部署，精度要求高 | AWQ |
| 追求极致精度（~FP16） | GPTQ |
| 小模型（<1B参数） | PTQ足够 |
| 大模型（>7B参数） | AWQ或GPTQ |
| 受限硬件（2-bit） | GPTQ |

---

## 总结

- **PTQ**：简单粗暴，直接量
- **AWQ**：看人下菜，重要的通道特殊照顾
- **GPTQ**：错位补偿，量化误差通过调整其他权重弥补

要点：

1. **按通道量化**：每个输出通道有独立的`scale`，通过`weight.abs().max(dim=1)`计算该通道的最大绝对值

2. **量化过程**：
   - `weight / scale` 将权重归一化到[-127, 127]
   - `round` 四舍五入
   - `clamp(-128, 127)` 确保在int8范围内
   - 转换为`torch.int8`类型

3. **存储为buffer**：使用`register_buffer`确保这些张量随模型保存和加载，但不会被优化器更新

4. **前向推理**：
   - 反量化：`int8.float() * scale` 恢复为fp32
   - 使用标准的矩阵乘法
   - 添加偏置（保持fp32）

5. **偏置处理**：偏置不量化，直接存储为fp32，因为偏置通常占比较小

这种训练后量化方法能有效减少模型大小（权重从fp32降到int8，减少75%），同时保持较高的精度。

In [ ]:
from typing import Optional

class Int8Linear(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: Optional[torch.Tensor] = None):
        """
        训练后量化的线性层，使用INT8权重
        
        Args:
            weight: 原始浮点权重，shape (out_features, in_features)
            bias: 偏置项，shape (out_features,)
        """
        super().__init__()
        
        # 保存原始shape信息
        self.out_features, self.in_features = weight.shape
        
        # 按通道量化（每个输出通道独立量化）
        # scale shape: (out_features, 1)
        self.register_buffer('scale', weight.abs().max(dim=1, keepdim=True)[0] / 127)
        
        # 量化权重到int8
        # 注意：需要先转换为float再计算，最后转为int8
        weight_int8 = torch.round(weight / self.scale)
        weight_int8 = weight_int8.clamp(-128, 127).to(torch.int8)
        self.register_buffer('weight_int8', weight_int8)
        
        # 处理偏置
        if bias is not None:
            # 偏置保持为fp32，直接存储
            self.register_buffer('bias', bias.clone())
        else:
            self.register_buffer('bias', None)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播：反量化然后矩阵乘法
        
        Args:
            x: 输入张量，shape (batch_size, in_features)
        
        Returns:
            输出张量，shape (batch_size, out_features)
        """
        # 反量化：int8 -> fp32 * scale
        # weight_fp32 shape: (out_features, in_features)
        weight_fp32 = self.weight_int8.float() * self.scale
        
        # 矩阵乘法
        out = torch.mm(x, weight_fp32.T)
        
        # 添加偏置
        if self.bias is not None:
            out = out + self.bias
        
        return out
    
    def extra_repr(self) -> str:
        """显示额外的信息"""
        return f'in_features={self.in_features}, out_features={self.out_features}'

In [ ]:
# 🧪 调试
w = torch.randn(8, 4)
q = Int8Linear(w)
x = torch.randn(2, 4)
print('输出:', q(x).形状)
print('dtype:', q.weight_int8.dtype)
print('最大量化误差:', (w - q.weight_int8.float() * q.scale).abs().max().item())

In [ ]:
# ✅ 提交
from torch_judge import check
check('int8_quantization')